# Cross-Database Validation: BrainMap

To ensure that our functional parcellation of the VMPFC is not an artifact of Neurosynth's automated coordinate-extraction framework, we perform an independent replication using the **BrainMap** database. BrainMap is a rigorously, manually curated meta-analytic database.

Using the `NiMARE` (Neuroimaging Meta-Analysis Research Environment) library, we aim to replicate our entire data-driven pipeline—from modeled activation generation to spatial clustering—and quantitatively compare the resulting topography against our primary findings.

In [ ]:
from pathlib import Path
import nibabel as nib
from utils.dice import max_dice_score
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
import joblib
from nilearn.image import resample_to_img
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
import matplotlib.pyplot as plt
import nilearn.plotting as nplt
from nimare.io import convert_sleuth_to_studyset
from nimare.io import convert_sleuth_to_dataset
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
import numpy as np
from nilearn.maskers import NiftiMasker
from scipy.interpolate import pchip_interpolate
from nimare.meta.kernel import ALEKernel, MKDAKernel


plt.rcParams['axes.grid'] = False
plt.rcParams['font.sans-serif'] = ['Arial']

DATA_PATH = Path('../data')
PLOTS_PATH = Path('../plots')
RESULTS_PATH = Path('../results')
PLOT_KWARGS_DICT = dict(dpi=300, transparent=True, bbox_inches='tight')
ROI_FILE = DATA_PATH / 'masks/VMPFC_mask_v2.nii'
unROI_FILE = DATA_PATH / 'masks/unVMPFC_mask_v2.nii'
ROI_IMAGE = nib.load(str(ROI_FILE))
unROI_IMAGE = nib.load(str(unROI_FILE))
RANDOM_STATE = 2026
N_CLUSTER_LIST = list(range(2, 7))

In [ ]:
collection = joblib.load(DATA_PATH / 'brainmap_data' / 'brainmap.dataset.pkl')
nimare_template = collection.masker.mask_img

ROI_IMAGE = resample_to_img(ROI_IMAGE, nimare_template, interpolation='nearest')
unROI_IMAGE = resample_to_img(unROI_IMAGE, nimare_template, interpolation='nearest')

ROI_masker = NiftiMasker(mask_img=ROI_IMAGE).fit()
unROI_masker = NiftiMasker(mask_img=unROI_IMAGE).fit()

# Modeled Activation Maps via MKDA

Unlike Neurosynth's pre-computed activation arrays, we must generate modeled activation maps from BrainMap's raw peak coordinates. To replicate the binary presence/absence logic of our primary pipeline, we employ a Multi-level Kernel Density Analysis (**MKDA**) kernel.

Each coordinate focus is expanded into a hard sphere with a radius of $8\text{ mm}$ (value = 1). We then extract these modeled maps using our previously defined VMPFC (`ROI`) and whole-brain (`unROI`) maskers. The resulting sparse matrices are explicitly binarized (`> 0`) to indicate the presence or absence of activation for each voxel across all BrainMap experiments.

In [ ]:
kernel = MKDAKernel(r=8, value=1)

ROI_masker = NiftiMasker(mask_img=ROI_IMAGE).fit()
roi_array = kernel.transform(collection, masker=ROI_masker, return_type="array").T
ROI_activation_raw = sp.csr_matrix(roi_array > 0, dtype=np.int8)

unROI_masker = NiftiMasker(mask_img=unROI_IMAGE).fit()
unroi_array = kernel.transform(collection, masker=unROI_masker, return_type="array").T
unROI_activation_raw = sp.csr_matrix(unroi_array > 0, dtype=np.int8)



print(f"ROI dataset: (n voxels, n experiments) = {ROI_activation_raw.shape}.")
print(f"unROI dataset: (n voxels, n experiments) = {unROI_activation_raw.shape}.")
print(f"ROI non-zero values: {np.unique(ROI_activation_raw.data)[:10]}")
print(f"unROI non-zero values: {np.unique(unROI_activation_raw.data)[:10]}")

ROI_studies_per_voxel = np.asarray(ROI_activation_raw.sum(axis=1)).ravel()
unROI_studies_per_voxel = np.asarray(unROI_activation_raw.sum(axis=1)).ravel()

# Dimensionality Reduction & Co-activation Matrix

We apply a strict inclusion threshold tailored to the BrainMap dataset size (`MIN_STUDIES = 120`), retaining only voxels with sufficient activation frequency.

Following the primary pipeline, we denoise the whole-brain data by extracting the top 100 Principal Components from the `unROI` matrix. The co-activation profile for each VMPFC voxel is then constructed by calculating its Pearson correlation distance against these whole-brain PCs.

In [ ]:
MIN_STUDIES = 120

ROI_voxels_to_use = ROI_studies_per_voxel >= MIN_STUDIES
unROI_voxels_to_use = unROI_studies_per_voxel >= MIN_STUDIES


ROI_activation = ROI_activation_raw[ROI_voxels_to_use].toarray().astype(np.float32)
unROI_activation = unROI_activation_raw[unROI_voxels_to_use].toarray().astype(np.float32)



print(f"ROI filtered: (n voxels, n experiments) = {ROI_activation.shape}.")
print(f"unROI filtered: (n voxels, n experiments) = {unROI_activation.shape}.")

In [ ]:
N_COMPONENTS = 100

pca = PCA(n_components=N_COMPONENTS, svd_solver="randomized", random_state=RANDOM_STATE)
unROI_activation_PCs = pca.fit_transform(unROI_activation.T).T

coactivation = pairwise_distances(
    ROI_activation,
    unROI_activation_PCs,
    metric="correlation",
    n_jobs=-1,
)

print(f"unROI dataset: (n PCs, n experiments) = {unROI_activation_PCs.shape}.")
print(f"coactivation: (n ROI voxels, n PCs) = {coactivation.shape}.")

# K-Means Clustering & Model Evaluation

With the co-activation distance matrix constructed, we partition the VMPFC into distinct functional subregions using **K-Means clustering**. To ensure stability and avoid local minima, we employ random initialization with a robust number of iterations (`n_init=1000`). We iterate this process across our target range of cluster sizes (from $K=2$ to $K=6$).

To quantitatively assess the quality of these partitions and determine the optimal cluster solution, we calculate two standard internal validation metrics for each $K$:
* **Silhouette Coefficient:** Evaluates how similar a voxel is to its own cluster compared to neighboring clusters (higher values indicate better defined clusters).
* **Davies-Bouldin Index:** Measures the ratio of within-cluster scatter to between-cluster separation (lower values indicate superior clustering).

Finally, we map the resulting array of cluster labels back into the original 3D neuroanatomical space. This allows us to visually inspect the topography for each value of $K$ and isolate the $K=3$ NIfTI image for our subsequent cross-database quantitative comparison.

In [ ]:

label_dict = {
    n_clusters: KMeans(
        n_clusters=n_clusters,
        init="random",
        n_init=1000,
        random_state=RANDOM_STATE,
    ).fit_predict(coactivation)
    + 1
    for n_clusters in N_CLUSTER_LIST
}


scores_dict = {
    "cluster_list": N_CLUSTER_LIST,
    "silhouette_scores": [
        silhouette_score(coactivation, label_dict[n_clusters])
        for n_clusters in N_CLUSTER_LIST
    ],
    "davies_bouldin_scores": [
        davies_bouldin_score(coactivation, label_dict[n_clusters])
        for n_clusters in N_CLUSTER_LIST
    ],
}


for n_clusters in N_CLUSTER_LIST:
    full_roi_labels = np.zeros(ROI_activation_raw.shape[0])
    full_roi_labels[ROI_voxels_to_use] = label_dict[n_clusters]
    img = ROI_masker.inverse_transform(full_roi_labels)
    nplt.plot_stat_map(
        img,
        title=f"K={n_clusters}",
        cmap="tab10",
        draw_cross=False,
        colorbar=False,
    )

    if  n_clusters == 3:
        img.to_filename(RESULTS_PATH / 'nii/BrainMap_K3.nii.gz')


In [ ]:
params = [('silhouette_scores', 'Silhouette Coefficient', max, '^', '#9B59B6', '#6c3f7f',
           (0.165, 0.23), (0.17, 0.19, 0.21,), (0, 0)),

          ('davies_bouldin_scores', 'Davies Bouldin Index', min, 'o', '#1ABC9C', '#128973',
           (1.5, 2.1), (1.6, 1.7, 1.8, 1.9, 2.0), (1, 1),), ]

fig, ax1 = plt.subplots(figsize=(2.5, 2))
ax2 = ax1.twinx()
axes = [ax1, ax2]

# plot scores: lines and points
x, d = N_CLUSTER_LIST, 0.5
for i, ax in enumerate(axes):
    (key, label, direction, marker, color, darkened_color, ylim, yticks, ax_cut_pos) = params[i]
    y = scores_dict[key]

    # plot smooth line
    x_smooth = np.linspace(x[0], x[-1], 500)
    y_smooth = pchip_interpolate(x, y, x_smooth)
    sns.lineplot(x=x_smooth, y=y_smooth, color=color, ax=ax, dashes=(2, 1), linewidth=1)

    # plot points
    kwargs = dict(ax=ax, zorder=2, linewidth=0.5, facecolor=color, edgecolor=color, marker=marker)
    # optimal_y = direction(y)
    # optimal_x = x[y.index(optimal_y)]
    sns.scatterplot(x=x, y=y, s=15, **kwargs)
    # sns.scatterplot(x=[optimal_x], y=[optimal_y], s=45, **kwargs)

    # set up axis
    ax.set_ylabel(label, color=darkened_color, fontsize=10)
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticks, color=darkened_color, fontsize=6)
    ax.set_ylim(ylim)
    ax.tick_params(axis='y', colors=darkened_color)

    # add cut in axis
    kwargs = dict(markersize=12, linestyle="none", color=darkened_color, mec=color, mew=1, clip_on=False)
    ax.plot(ax_cut_pos, [0.02, 0.05], marker=[(-1, d), (1, -d)], transform=ax2.transAxes, **kwargs)

# plot settings
ax2.spines['left'].set_color(params[0][5])
ax2.spines['left'].set_linewidth(2)
ax1.tick_params(axis='y', width=2)

ax2.spines['right'].set_color(params[1][5])
ax2.spines['right'].set_linewidth(2)
ax2.tick_params(axis='y', width=2)

ax2.spines['bottom'].set_color('black')

ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)

ax1.spines['bottom'].set_visible(False)
ax2.spines['bottom'].set_visible(False)

ax2.set_xticks([])
ax2.set_xticklabels([], fontsize=12)
# ax2.set_xlim(1.2, 6.8)

# # save figure
fig.savefig(PLOTS_PATH / 'scores_BrainMap.svg', **PLOT_KWARGS_DICT )
fig.savefig(PLOTS_PATH / 'scores_BrainMap.png', **PLOT_KWARGS_DICT)

## Quantitative Agreement Across Databases

Visual similarity is subjective. To strictly quantify the reproducibility of our parcellation, we compute the **Dice Similarity Coefficient (DSC)** between the original Neurosynth-derived $K=3$ map and this newly generated BrainMap-derived $K=3$ map. A high Dice score here provides compelling evidence that the functional subdivisions of the VMPFC represent robust, database-independent neurobiological architectures.

In [ ]:
k = 3
data_1_raw = nib.load(RESULTS_PATH / f'nii/K{k}.nii.gz').get_fdata()

full_roi_labels = np.zeros(ROI_activation_raw.shape[0])
full_roi_labels[ROI_voxels_to_use] = label_dict[k]
img = ROI_masker.inverse_transform(full_roi_labels)
data_2_raw = img.get_fdata()

mask = (data_1_raw > 0) & (data_2_raw > 0)
score = max_dice_score(data_1_raw[mask], data_2_raw[mask], k)
print(f'k={k}, score={score:.4f}')